# RAG & Knowledge Systems: Vector Storage Essentials

Once we have our chunks, we need a place to store them that allows for lightning-fast "semantic search." This is where **Vector Databases** come in.

## 1. What is a Vector Database?
Traditional databases (SQL) are good for exact matches (e.g., `WHERE name = 'Bala'`). Vector databases (Chroma, FAISS) are built for **Similarity Matches** (e.g., `FIND text WHERE meaning IS SIMILAR TO 'How to build agents?'`).

---

## 2. Environment Setup
We will use **Chroma** (persistent) and **FAISS** (in-memory) with **Hugging Face** embeddings.

In [7]:
import os
import shutil
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

load_dotenv()
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=model_name)

print("Vector stores & Local Embeddings ready!")

Vector stores & Local Embeddings ready!


## 3. Creating Documents
Let's create some sample documents to store.

In [8]:
docs = [
    Document(page_content="The first rule of Agentic AI is stability.", metadata={"id": 1}),
    Document(page_content="Memory management is crucial for long conversations.", metadata={"id": 2}),
    Document(page_content="RAG allows agents to access external knowledge bases.", metadata={"id": 3}),
    Document(page_content="Deeplearning.ai offers great courses on AI.", metadata={"id": 4})
]

## 4. Using ChromaDB (Persistent Storage)
Chroma is excellent for production because it can save your vectors to the disk so you don't have to re-embed every time.

In [9]:
CHROMA_PATH = "chroma_db"

# Clean up previous database if it exists
if os.path.exists(CHROMA_PATH):
    shutil.rmtree(CHROMA_PATH)

# Create the vector store
db = Chroma.from_documents(
    docs, 
    embeddings, 
    persist_directory=CHROMA_PATH
)

print(f"Database persisted to {CHROMA_PATH}")

InternalError: Query error: Database error: error returned from database: (code: 1032) attempt to write a readonly database

## 5. Similarity Search in Chroma
Let's query our database.

In [ ]:
query = "How do agents access knowledge?"
results = db.similarity_search(query, k=2)

for i, res in enumerate(results):
    print(f"Result {i+1}: {res.page_content} (ID: {res.metadata['id']})")

Result 1: RAG allows agents to access external knowledge bases. (ID: 3)
Result 2: The first rule of Agentic AI is stability. (ID: 1)


## 6. Similarity Search with Scores
Sometimes you want to know *how sure* the database is about the match. Lower score usually means more similar (distance).

In [ ]:
results_with_scores = db.similarity_search_with_score(query, k=2)
for doc, score in results_with_scores:
    print(f"Score: {score:.4f} | Content: {doc.page_content}")

Score: 0.6427 | Content: RAG allows agents to access external knowledge bases.
Score: 1.1157 | Content: The first rule of Agentic AI is stability.


## 7. Using FAISS (In-Memory Storage)
FAISS (Facebook AI Similarity Search) is incredibly fast and great for large-scale datasets, often kept in memory for maximum speed.

In [ ]:
faiss_db = FAISS.from_documents(docs, embeddings)
faiss_results = faiss_db.similarity_search("Tell me about memory.")

print(f"FAISS Result: {faiss_results[0].page_content}")

ImportError: Could not import faiss python package. Please install it with `pip install faiss-gpu` (for CUDA supported GPU) or `pip install faiss-cpu` (depending on Python version).

## 8. Metadata Filtering
What if you only want to search through documents from a specific source? Metadata filtering allows you to combine SQL-like filters with vector search.

In [ ]:
filtered_results = db.similarity_search(
    "agent knowledge",
    k=1,
    filter={"id": 3}
)
print(f"Filtered Result: {filtered_results[0].page_content}")

## 9. Retriever Abstraction
In LangChain, we rarely use the DB object directly. We wrap it in a **Retriever** interface.

In [ ]:
retriever = db.as_retriever(search_kwargs={"k": 1})
retrieved_docs = retriever.invoke("RAG")
print(f"Retrieved via Interface: {retrieved_docs[0].page_content}")

## 10. Summary & Pro-Tips
1. **Local Embeddings Advantage**: Since we switched to Hugging Face, we no longer need an active internet connection or API quota for the retrieval part of RAG.
2. **Chroma persistence**: Remember that your `chroma_db` folder contains all your data; you can copy it to another project and load it instantly.
3. **FAISS Scalability**: For billions of vectors, FAISS specialized index types (like HNSW) are state-of-the-art.

---